<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c40-header.png" alt="Nextia Learning · Experiments and Causal Impact: Did Our Change Work?" width="100%">

# Report a result people trust

**[Report a result people trust](https://learning.nextia-ai.com/courses/experiments/m08/report-a-result-people-trust/)** · Experiments and Causal Impact: Did Our Change Work? · Module 8, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compute a test's effect and guardrail with their intervals, write the decision rules of a test plan as one small function, and turn the numbers into a short, honest result report.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | The test plan, from [Write the plan first](https://learning.nextia-ai.com/courses/experiments/m02/write-the-plan-first/). |
| **Tested** | Python 3.12 with pandas 3.1.1, SciPy 1.15.3, statsmodels 0.15.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/report-a-result-people-trust/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C40/M08-L03-report-a-result-people-trust/report-a-result-people-trust-solution.ipynb).

## Setup: install the packages

Run the next cell first. It installs the versions that the course uses: pandas 3.1.1, SciPy 1.15.3 and statsmodels 0.15.0, and matplotlib 3.11.2 for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with these versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scipy==1.15.3 statsmodels==0.15.0 matplotlib==3.11.2
import importlib.metadata as _md
for _name in ["pandas", "scipy", "statsmodels", "matplotlib"]:
    print(_name, _md.version(_name))

## Setup: get the practice data

Run the next cell. It downloads the simulator's data for this lesson (less than 3 MB) into a folder `larkfield-experiments/data`, and checks each file's checksum. It needs an internet connection and no account.

You should see `Ready: 2 files in data/.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-experiments` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: the course simulator made it. It has no real people. Each folder has `truth.json`: the true effects that the simulator knows. A real test never has this file.

In [ ]:
# Setup: download the course data into larkfield-experiments/data/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C40/data/"
FILES = {
    "delivery/customers.csv.gz": "c22521f272d36b36fd26234d53d04aac1c8be347a5014ee3e2aeeffcc34a0c19",
    "delivery/truth.json": "6f088f29a112de123bb1da3bac0940a28c5adb706e9f24007686c373c14ab6a4"
}
LOCAL = os.environ.get("NX_DATA_LOCAL")  # authors only: a local copy of the labs data
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("larkfield-experiments").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    if LOCAL:
        shutil.copy(Path(LOCAL) / name, path)
    else:
        download(DATA + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-experiments and run again.")
os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in data/.")

## Setup: helpers

The next cell loads pandas as `pd`, NumPy as `np`, SciPy's `stats` and statsmodels' formula interface as `smf`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `truth(name)` reads a scenario's `truth.json`: the simulator's true effects.

Run the cell. It prints nothing.

In [ ]:
# Helpers: checks that do not show the answer, and the simulator's truth.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
pd.set_option("display.precision", 4)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def truth(name):
    """The simulator's true effects for a scenario (a real test never has this)."""
    return json.loads(open(f"data/{name}/truth.json").read())

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The effect and the guardrail

Hana's free-delivery test: revenue per customer (the primary metric) and the return rate, returns per order (the guardrail). The return rate is a ratio metric, so it uses the delta method.

In [ ]:
df = pd.read_csv("data/delivery/customers.csv.gz")
a, b = df[df.variant == "A"], df[df.variant == "B"]

def welch(xa, xb):
    d = xb.mean() - xa.mean()
    va, vb = xa.var(ddof=1) / len(xa), xb.var(ddof=1) / len(xb)
    se = np.sqrt(va + vb)
    dof = (va + vb) ** 2 / (va**2 / (len(xa) - 1) + vb**2 / (len(xb) - 1))
    t = stats.t.ppf(0.975, dof)
    return {"difference": d, "low": d - t * se, "high": d + t * se}

def ratio(num_a, den_a, num_b, den_b):
    def r_and_var(num, den):
        r = num.mean() / den.mean()
        c = np.cov(num, den)
        return r, (c[0, 0] - 2 * r * c[0, 1] + r**2 * c[1, 1]) / (den.mean() ** 2 * len(num))
    (ra, va), (rb, vb) = r_and_var(num_a, den_a), r_and_var(num_b, den_b)
    d, se = rb - ra, np.sqrt(va + vb)
    return {"difference": d, "low": d - 1.96 * se, "high": d + 1.96 * se}

revenue = welch(a.revenue, b.revenue)
oa, ob = a[a.orders > 0], b[b.orders > 0]
returns = ratio(oa.returns, oa.orders, ob.returns, ob.orders)
print(f"revenue {revenue['difference']:+.4f} ({revenue['low']:+.4f} to {revenue['high']:+.4f})")
print(f"return rate {returns['difference']:+.4f} ({returns['low']:+.4f} to {returns['high']:+.4f})")

> **Check.** You should see `revenue +1.0266 (+0.1583 to +1.8950)` and `return rate +0.0129 (+0.0068 to +0.0190)`, as `xp report delivery` writes.

## 2. The decision rules

The plan's rules: if the interval of the primary metric is above 0, ship B, unless a guardrail is clearly worse (its interval is above 0): then discuss. Below 0: keep A. Crossing 0: no clear effect. (The sample ratio check comes before all of this; this test passed it.)

In [ ]:
def decide(primary, guardrail):
    if primary["low"] > 0:
        return "Discuss before shipping" if guardrail["low"] > 0 else "Ship B"
    if primary["high"] < 0:
        return "Keep A"
    return "No clear effect: keep A"

print(decide(revenue, returns))

> **Check.** You should see `Discuss before shipping`.

## 3. Two tests with "no clear effect"

Two made-up tests with the same estimate, −0.05 dollars. Hana's break-even is +1.25 dollars.

In [ ]:
no_guardrail_problem = {"low": -1.0, "high": -0.5}
test_x = {"difference": -0.05, "low": -0.30, "high": 0.20}
test_y = {"difference": -0.05, "low": -2.00, "high": 1.90}
for name, t in (("X", test_x), ("Y", test_y)):
    rules_out_break_even = t["high"] < 1.25
    print(name, decide(t, no_guardrail_problem), "| rules out the break-even:", rules_out_break_even)

> **Check.** You should see both tests with `No clear effect: keep A`, test X with `True` and test Y with `False`.

The decision is the same. The message is not: test X answers the question, test Y does not.

## 4. Your turn: say "no clear effect" well

> **Your turn.** Write the sentence for test Y in `sentence_y`: the decision, the range with both ends (written as -2.00 and +1.90), and what it means. The check looks only for the two ends of the range; the words are yours.

In [ ]:
sentence_y = ...  # for example: "No clear effect. The change is between ... and ... dollars per customer: ..."

In [ ]:
expect('sentence_y', None if sentence_y is ... else all(x in str(sentence_y) for x in ("2.00", "1.90")), True)

## Next

The [lesson page](https://learning.nextia-ai.com/courses/experiments/m08/report-a-result-people-trust/) has the one-page report, how Priya edits the tool's draft, and the knowledge checks that count toward your certificate. In the project: `xp report delivery` and `xp report srm`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Experiments as a team habit](https://learning.nextia-ai.com/courses/experiments/m08/experiments-as-a-team-habit/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/experiments/m08/report-a-result-people-trust/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.